In [ ]:
# Import the Google Colab library to access Google Drive
from google.colab import drive
# Mount Google Drive in the Colab environment to access stored files
drive.mount('/gdrive')
# Change the current working directory to the Google Drive location
%cd /gdrive

Mounted at /gdrive
/gdrive


In [ ]:
#Importing libraries
import os
import numpy as np
from skimage import io, transform
from skimage.io import imsave
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate, Activation, Dropout, Add
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import KFold

In [ ]:
# Residual block used in the neural network
def res_block(x, filters):
    shortcut = x

    x = Conv2D(filters, 3, padding="same", kernel_initializer="he_normal")(x)
    x = Activation("relu")(x)
    x = Conv2D(filters, 3, padding="same", kernel_initializer="he_normal")(x)

    # Adjustment if channels change
    if shortcut.shape[-1] != filters:
        shortcut = Conv2D(filters, 1, padding="same")(shortcut)

    x = Add()([x, shortcut])
    x = Activation("relu")(x)
    return x

In [ ]:
# Definition of the ResUNet architecture
def ResUNet(pretrained_weights = None, input_size=(256,256,1)):
    inputs = Input(input_size)

    # Encoder
    c1 = res_block(inputs, 64)
    p1 = MaxPooling2D((2,2))(c1)

    c2 = res_block(p1, 128)
    p2 = MaxPooling2D((2,2))(c2)

    c3 = res_block(p2, 256)
    p3 = MaxPooling2D((2,2))(c3)

    c4 = res_block(p3, 512)
    d4 = Dropout(0.5)(c4)
    p4 = MaxPooling2D((2,2))(d4)

    # Bottleneck
    c5 = res_block(p4, 1024)

    # Decoder
    u6 = Conv2D(512, 2, padding='same')(UpSampling2D((2,2))(c5))
    m6 = concatenate([d4, u6])
    c6 = res_block(m6, 512)

    u7 = Conv2D(256, 2, padding='same')(UpSampling2D((2,2))(c6))
    m7 = concatenate([c3, u7])
    c7 = res_block(m7, 256)

    u8 = Conv2D(128, 2, padding='same')(UpSampling2D((2,2))(c7))
    m8 = concatenate([c2, u8])
    c8 = res_block(m8, 128)

    u9 = Conv2D(64, 2, padding='same')(UpSampling2D((2,2))(c8))
    m9 = concatenate([c1, u9])
    c9 = res_block(m9, 64)

    #Output layer
    outputs = Conv2D(1, 1, activation='sigmoid')(c9)

    model = Model(inputs, outputs)
    return model

    if(pretrained_weights):
    	model.load_weights(pretrained_weights)
    return model

In [ ]:
# Definition of image and mask paths
img_path  = '/gdrive/My Drive/RedDataBase/images/'
mask_path = '/gdrive/My Drive/RedDataBase/labels/'
# Obtains and alphabetically sorts the image names
img_names  = sorted(os.listdir(img_path))
mask_names = sorted(os.listdir(mask_path))

In [ ]:
# Image loading and preprocessing
X, Y = [], []

for img_name, mask_name in zip(img_names, mask_names):

    img  = io.imread(img_path + img_name, as_gray=True)
    mask = io.imread(mask_path + mask_name, as_gray=True)

    img  = transform.resize(img, (256,256))
    mask = transform.resize(mask, (256,256))

    img  = img / 255.0
    mask = (mask > 0.5).astype(np.float32)

    # Adds an aditional channel for compatibility with TensorFlow/Keras
    X.append(img[..., np.newaxis])
    Y.append(mask[..., np.newaxis])

X = np.array(X)
Y = np.array(Y)

# Show the final dimensions of the datasets
print(X.shape, Y.shape)

(226, 256, 256, 1) (226, 256, 256, 1)


In [ ]:
# Data augmentation configuration
data_gen_args = dict(
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest')

#Data generator for images and masks
image_datagen = ImageDataGenerator(**data_gen_args)
mask_datagen  = ImageDataGenerator(**data_gen_args)

#Training generation
def train_generator(X, Y, batch_size, seed=1):

    img_gen = image_datagen.flow(X, batch_size=batch_size, seed=seed)
    mask_gen = mask_datagen.flow(Y, batch_size=batch_size, seed=seed)

    while True:
        yield next(img_gen), next(mask_gen)

In [ ]:
# Function to save predicted segmentations
def save_segmentations(model, X, Y, names_val,  fold, save_dir, threshold=0.5):

    fold_dir = os.path.join(save_dir, f'fold_{fold}')
    os.makedirs(fold_dir, exist_ok=True)

    preds = model.predict(X)

    for i in range(len(X)):


        pr  = preds[i,:,:,0]
        prb = (pr > threshold).astype(np.uint8)
        name = os.path.splitext(os.path.basename(names_val[i]))[0]

        imsave(f'{fold_dir}/{name}_pred_bin.png', prb*255)

In [ ]:
# K-Fold cross-validation configuration
img_names = np.array(img_names)
kf = KFold(n_splits=5, shuffle=True, random_state=1)

batch_size = 10
epochs = 98
acc_scores = []
precision_scores = []
recall_scores = []
fold = 1

# K-Fold training and validation
for train_idx, val_idx in kf.split(X):

    print(f"\n FOLD {fold}")

    X_train, X_val = X[train_idx], X[val_idx]
    Y_train, Y_val = Y[train_idx], Y[val_idx]
    names_val = img_names[val_idx]

    # Model construction and compilation
    model = ResUNet(input_size=(256,256,1))
    model.compile(
        optimizer=Adam(1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy', Precision(name='precision'),
        Recall(name='recall')])

    # Callback to save the best model
    checkpoint = ModelCheckpoint(
        f'/gdrive/My Drive/RedDataBase/resunet_fold_{fold}.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1)

    # Training generator
    train_gen = train_generator(X_train, Y_train, batch_size)
    steps = len(X_train) // batch_size

    # Model training
    history = model.fit(
        train_gen,
        steps_per_epoch=steps,
        validation_data=(X_val, Y_val),
        epochs=epochs,
        callbacks=[checkpoint],
        verbose=1)

    best_acc = max(history.history['val_accuracy'])
    acc_scores.append(best_acc)
    best_precision = max(history.history['val_precision'])
    best_recall = max(history.history['val_recall'])
    precision_scores.append(best_precision)
    recall_scores.append(best_recall)

    # Load best model and save segmentations
    best_model = ResUNet(input_size=(256,256,1))
    best_model.compile(
        optimizer=Adam(1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy', Precision(name='precision'),
        Recall(name='recall')])
    best_model.load_weights(
        f'/gdrive/My Drive/RedDataBase/resunet_fold_{fold}.keras')

    save_segmentations(best_model,X_val,Y_val,names_val,fold,
        save_dir='/gdrive/My Drive/RedDataBase/predict')

    print(f"Best accuracy fold {fold}: {best_acc:.4f}")

    fold += 1


 FOLD 1
Epoch 1/98
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.7410 - loss: 0.5074 - precision: 0.1036 - recall: 0.1923
Epoch 1: val_accuracy improved from None to 0.90691, saving model to /gdrive/My Drive/RedDataBase/resunet_fold_1.keras

Epoch 1: finished saving model to /gdrive/My Drive/RedDataBase/resunet_fold_1.keras
18/18 ━━━━━━━━━━━━━━━━━━━━ 59s 1s/step - accuracy: 0.8484 - loss: 0.4094 - precision: 0.1036 - recall: 0.0530 - val_accuracy: 0.9069 - val_loss: 0.3083 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/98
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.8913 - loss: 0.3216 - precision: 0.0000e+00 - recall: 0.0000e+00
Epoch 2: val_accuracy did not improve from 0.90691
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 88ms/step - accuracy: 0.8946 - loss: 0.3079 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.9069 - val_loss: 0.3300 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 3/98
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 130 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step
Best accuracy fold 1: 0.9681

 FOLD 2
Epoch 1/98
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.7393 - loss: 0.5261 - precision: 0.0878 - recall: 0.1859
Epoch 1: val_accuracy improved from None to 0.89187, saving model to /gdrive/My Drive/RedDataBase/resunet_fold_2.keras

Epoch 1: finished saving model to /gdrive/My Drive/RedDataBase/resunet_fold_2.keras
18/18 ━━━━━━━━━━━━━━━━━━━━ 28s 662ms/step - accuracy: 0.8504 - loss: 0.4323 - precision: 0.0878 - recall: 0.0475 - val_accuracy: 0.8919 - val_loss: 0.3512 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/98
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.9014 - loss: 0.3171 - precision: 0.0000e+00 - recall: 0.0000e+00
Epoch 2: val_accuracy did not improve from 0.89187
18/18 ━━━━━━━━━━━━━━━━━━━━ 15s 88ms/step - accuracy: 0.9007 - loss: 0.3095 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.8919 - val_loss: 0.3399 - val_precision: 0.0000e+00 - val_recall: 0.0000e

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 130 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step
Best accuracy fold 2: 0.9605

 FOLD 3
Epoch 1/98
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - accuracy: 0.8711 - loss: 0.4971 - precision: 0.0496 - recall: 0.0110
Epoch 1: val_accuracy improved from None to 0.91694, saving model to /gdrive/My Drive/RedDataBase/resunet_fold_3.keras

Epoch 1: finished saving model to /gdrive/My Drive/RedDataBase/resunet_fold_3.keras
18/18 ━━━━━━━━━━━━━━━━━━━━ 24s 405ms/step - accuracy: 0.8841 - loss: 0.4164 - precision: 0.0496 - recall: 0.0030 - val_accuracy: 0.9169 - val_loss: 0.3370 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/98
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - accuracy: 0.8944 - loss: 0.3078 - precision: 0.0000e+00 - recall: 0.0000e+00
Epoch 2: val_accuracy did not improve from 0.91694
18/18 ━━━━━━━━━━━━━━━━━━━━ 10s 98ms/step - accuracy: 0.8934 - loss: 0.3065 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.9169 - val_loss: 0.3361 - val_precision: 0.0000e+00 - val_recall: 0.0000e

...

2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 2s/step


/usr/local/lib/python3.12/dist-packages/skimage/_shared/utils.py:328: UserWarning: /gdrive/My Drive/RedDataBase/predict/fold_4/WT_091_OriginalG_pred_bin.png is a low contrast image
  return func(*args, **kwargs)


Best accuracy fold 4: 0.9646

 FOLD 5
Epoch 1/98
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.8399 - loss: 0.4912 - precision: 0.0966 - recall: 0.0567
Epoch 1: val_accuracy improved from None to 0.90402, saving model to /gdrive/My Drive/RedDataBase/resunet_fold_5.keras

Epoch 1: finished saving model to /gdrive/My Drive/RedDataBase/resunet_fold_5.keras
18/18 ━━━━━━━━━━━━━━━━━━━━ 23s 402ms/step - accuracy: 0.8793 - loss: 0.3963 - precision: 0.0966 - recall: 0.0160 - val_accuracy: 0.9040 - val_loss: 0.3076 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/98
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.9027 - loss: 0.3466 - precision: 0.0000e+00 - recall: 0.0000e+00
Epoch 2: val_accuracy did not improve from 0.90402
18/18 ━━━━━━━━━━━━━━━━━━━━ 9s 87ms/step - accuracy: 0.8950 - loss: 0.3322 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.9040 - val_loss: 0.3078 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 3/98
18/18 ━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 130 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step
Best accuracy fold 5: 0.9686


In [ ]:
# Cálculo de resultados finales
acc_scores = np.array(acc_scores)
precision_scores = np.array(precision_scores)
recall_scores = np.array(recall_scores)

print("\n RESULTADOS FINALES K-FOLD")
print(f"Accuracy media : {acc_scores.mean():.4f}")
print(f"Std accuracy   : {acc_scores.std():.4f}")

print(f"Precision media: {precision_scores.mean():.4f}")
print(f"Std precision  : {precision_scores.std():.4f}")

print(f"Recall media   : {recall_scores.mean():.4f}")
print(f"Std recall     : {recall_scores.std():.4f}")


 RESULTADOS FINALES K-FOLD
Accuracy media : 0.9654
Std accuracy   : 0.0029
Precision media: 0.9415
Std precision  : 0.0361
Recall media   : 0.8122
Std recall     : 0.0185


In [ ]:

# Load best global model

best_fold = np.argmax(acc_scores) + 1

print("Mejor fold:", best_fold)
print("Mejor accuracy:", acc_scores[best_fold - 1])

best_model = ResUNet(input_size=(256,256,1))

best_model.compile(optimizer=Adam(1e-4),loss='binary_crossentropy',
    metrics=['accuracy',Precision(name='precision'),Recall(name='recall')])

best_model.load_weights(f'/gdrive/My Drive/RedDataBase/resunet_fold_{best_fold}.keras')

print("Modelo cargado correctamente")

Mejor fold: 5
Mejor accuracy: 0.9686238765716553
Modelo cargado correctamente


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 130 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
# Load external test

external_img_path  = '/gdrive/My Drive/RedDataBase/Validación/images/'
external_mask_path = '/gdrive/My Drive/RedDataBase/Validación/labels/'

external_img_names  = sorted(os.listdir(external_img_path))
external_mask_names = sorted(os.listdir(external_mask_path))

X_external = []
Y_external = []

for img_name, mask_name in zip(external_img_names, external_mask_names):

    img = io.imread(os.path.join(external_img_path, img_name), as_gray=True)
    mask = io.imread(os.path.join(external_mask_path, mask_name), as_gray=True)

    img = transform.resize(img, (256,256))
    mask = transform.resize(mask, (256,256))

    img = img / 255.0
    mask = (mask > 0.5).astype(np.float32)

    X_external.append(img[..., np.newaxis])
    Y_external.append(mask[..., np.newaxis])

X_external = np.array(X_external)
Y_external = np.array(Y_external)

print("Imágenes externas:", X_external.shape)
print("Máscaras externas:", Y_external.shape)

Imágenes externas: (56, 256, 256, 1)
Máscaras externas: (56, 256, 256, 1)


In [ ]:
# Evaluate external test
results = best_model.evaluate(X_external, Y_external, verbose=1)

print("\n RESULTADOS TEST EXTERNO")
for name, value in zip(best_model.metrics_names, results):
    print(f"{name}: {value:.4f}")

2/2 ━━━━━━━━━━━━━━━━━━━━ 11s 8s/step - accuracy: 0.9572 - loss: 0.1202 - precision: 0.8462 - recall: 0.7631

 RESULTADOS TEST EXTERNO
loss: 0.1202
compile_metrics: 0.9572


In [ ]:
# Save external segmentations

external_save_dir = '/gdrive/My Drive/RedDataBase/Validación/predict/'
os.makedirs(external_save_dir, exist_ok=True)

preds_external = best_model.predict(X_external)

threshold = 0.5

for i in range(len(X_external)):

    pr = preds_external[i,:,:,0]
    prb = (pr > threshold).astype(np.uint8)

    name = os.path.splitext(os.path.basename(external_img_names[i]))[0]

    imsave(os.path.join(external_save_dir, f'{name}_pred_bin.png'),prb * 255)

print("Segmentaciones externas guardadas en:")
print(external_save_dir)

2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step
Segmentaciones externas guardadas en:
/gdrive/My Drive/RedDataBase/Validación/predict/
